In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "groq:openai/gpt-oss-20b",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [6]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='485d6129-4877-4f83-888a-8d42ad5f047b'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function update_favourite_colour with favourite_colour: "green".', 'tool_calls': [{'id': 'fc_7eaebd22-7cae-4ae6-9e8a-f4ed8a0348de', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 137, 'total_tokens': 182, 'completion_time': 0.047039539, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.007850597, 'prompt_tokens_details': None, 'queue_time': 0.218517127, 'total_time': 0.054890136}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_66f3850a1a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provi

In [7]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='2c3b1f55-8405-4311-aa5c-9dc32d51f0ef'),
              AIMessage(content='Hello! I’m doing great—thanks for asking. How can I help you today?', additional_kwargs={'reasoning_content': 'User says hello. We respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 138, 'total_tokens': 174, 'completion_time': 0.037801194, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.007882298, 'prompt_tokens_details': None, 'queue_time': 0.20948805, 'total_time': 0.045683492}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5979a0e1b7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a09a82-4da3-7f11-b361-b56c4200bba1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 138, 'output_tokens': 36, 'total_tokens'

## Read state

In [8]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "groq:openai/gpt-oss-20b",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [9]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='32b3f0bc-22ea-41fe-bd66-39806e3ef4a0'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'User says: "My favourite colour is green". We should update favourite colour in state using function update_favourite_colour with favourite_colour: "green".', 'tool_calls': [{'id': 'fc_a7f2e377-8e20-4bed-b768-0dbc94ffced5', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 161, 'total_tokens': 219, 'completion_time': 0.060397856, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.009061185, 'prompt_tokens_details': None, 'queue_time': 0.149394708, 'total_time': 0.069459041}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8e23cedc90', 'service_tier': 'o

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='32b3f0bc-22ea-41fe-bd66-39806e3ef4a0'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'User says: "My favourite colour is green". We should update favourite colour in state using function update_favourite_colour with favourite_colour: "green".', 'tool_calls': [{'id': 'fc_a7f2e377-8e20-4bed-b768-0dbc94ffced5', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 161, 'total_tokens': 219, 'completion_time': 0.060397856, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.009061185, 'prompt_tokens_details': None, 'queue_time': 0.149394708, 'total_time': 0.069459041}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8e23cedc90', 'service_tier': 'o

In [11]:
print(response["messages"][-1].content)

Your favourite colour is green.
